# Bahraini Model Baseline Test — Whisper Small (CPU)

**Goal:** measure the baseline performance (no fine-tuning) of `openai/whisper-small` on the test split of `Hishambarakat/Bahraini_Speech_Dataset`.

**Note:** this notebook is adapted to run without a GPU. The 4-bit quantization (which needs a GPU + bitsandbytes) has been removed, and we use `whisper-small` instead of `large-v3` since it's much lighter and runs in a reasonable time on CPU.

> Security heads-up: in the original screenshot you shared, your HuggingFace token was hardcoded in the notebook (`login("hf_...")`). That token is now exposed — revoke it from your HF account settings and generate a new one, and avoid pasting tokens directly into code. Use `login()` interactively instead (it will prompt you to paste the token at runtime).

## 1. Installing & Importing Libraries

In [3]:
# Run in the terminal, or uncomment and run here (CPU build of torch)
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# %pip install "datasets<4.0.0" soundfile librosa
# %pip install evaluate jiwer accelerate transformers tqdm

In [2]:
import gc
import torch
from datasets import load_from_disk, Audio
from huggingface_hub import snapshot_download, login
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import evaluate
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Don't hardcode the token in code — call login() and paste it interactively
# login()

## 2. Device Verification (CPU/GPU)

In [4]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available, running on CPU (slower, but whisper-small handles it fine)")

Device: cpu
No GPU available, running on CPU (slower, but whisper-small handles it fine)


## 3. Dataset Loading & Cleaning

In [5]:
repo_id = "Hishambarakat/Bahraini_Speech_Dataset"
local_path = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["dataset_dict.json", "dataset_info.json", "train/**", "test/**"],
    max_workers=1,      # forces sequential downloading to prevent Jupyter from freezing
    resume_download=True
)

dataset = load_from_disk(local_path)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print("Train clips:", len(dataset["train"]))
print("Test clips:", len(dataset["test"]))

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 37 files:   0%|          | 0/37 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/30 [00:00<?, ?it/s]

Train clips: 87708
Test clips: 2713


In [6]:
# Filter out empty text and audio clips shorter than 0.5s (8000 samples at 16kHz)
def is_valid(sample):
    valid_text = sample["text"] is not None and len(sample["text"].strip()) > 0
    valid_audio = len(sample["audio"]["array"]) >= 8000
    return valid_text and valid_audio

dataset = dataset.filter(is_valid, num_proc=4)
print("Cleaned Train:", len(dataset["train"]))
print("Cleaned Test:", len(dataset["test"]))

Filter (num_proc=4):   0%|          | 0/87708 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/2713 [00:00<?, ? examples/s]

Cleaned Train: 87371
Cleaned Test: 2695


## 4. Model & Processor Initialization (whisper-small, no quantization)

In [7]:
model_id = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    model_id,
    language="Arabic",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.to(device)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.eval()

print("Whisper-small loaded successfully on", device)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Whisper-small loaded successfully on cpu


C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--openai--whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 5. Test Data Preprocessing

In [8]:
def prepare_dataset(batch, processor):
    audio = batch["audio"]
    # Extract audio features
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # Tokenize target text
    batch["labels"] = processor.tokenizer(
        batch["text"], truncation=True, max_length=448
    ).input_ids
    return batch

# CPU is much slower than GPU, so start with a reasonable sample count (bump it up later)
TEST_SAMPLES = 100

test_sub = dataset["test"].select(range(min(TEST_SAMPLES, len(dataset["test"]))))

test_processed = test_sub.map(
    prepare_dataset,
    fn_kwargs={"processor": processor},
    remove_columns=dataset["test"].column_names
)

print("Test samples ready:", len(test_processed))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Test samples ready: 100


## 6. Baseline Evaluation (WER & CER)

In [9]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [10]:
def collate_fn(features):
    input_features = [{"input_features": f["input_features"]} for f in features]
    return processor.feature_extractor.pad(input_features, return_tensors="pt")

BATCH_SIZE = 2  # keep it small since we're on CPU

dataloader = DataLoader(test_processed, batch_size=BATCH_SIZE, collate_fn=collate_fn)

references = [
    processor.tokenizer.decode(lbl, skip_special_tokens=True)
    for lbl in test_processed["labels"]
]

predictions = []
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Evaluating Whisper-Small (CPU)"):
        inputs = batch["input_features"].to(device)
        pred_ids = model.generate(
            inputs, language="arabic", task="transcribe", max_new_tokens=225
        )
        preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
        predictions.extend(preds)

base_wer = 100 * wer_metric.compute(predictions=predictions, references=references)
base_cer = 100 * cer_metric.compute(predictions=predictions, references=references)

print("\n--- Whisper-Small Baseline Results ---")
print(f"WER: {base_wer:.2f}%")
print(f"CER: {base_cer:.2f}%")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Evaluating Whisper-Small (CPU):   0%|          | 0/50 [00:00<?, ?it/s]

[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `


--- Whisper-Small Baseline Results ---
WER: 92.03%
CER: 66.24%


## 7. Single Sample Inference Test (Sanity Check)

In [12]:
sample = dataset["test"][9]
audio = sample["audio"]["array"]
ground_truth = sample["text"]

inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)

with torch.no_grad():
    pred_id = model.generate(inputs, language="arabic", task="transcribe")
    pred_text = processor.batch_decode(pred_id, skip_special_tokens=True)[0]

print("Truth:", ground_truth)
print("Prediction:", pred_text)

Truth: فضل رب العالمين أكيد أولًا وآخرًا لأن الموضوع حتى دي يحتاج أن
Prediction:  فضرب العالمين أكيد أولاً وأخراً لأن الموضوع حتى دي يحتاجنا الأسلحة
